# Fine-tuning Qwen3-1.7B avec Unsloth (Google Colab)

Ce notebook fine-tune **Qwen3-1.7B** sur des datasets médicaux SFT et DPO préparés dans le notebook EDA.

**Pré-requis :**
- Runtime GPU (T4 gratuit suffit, Qwen3-1.7B en 4-bit ≈ 3 Go VRAM)
- Datasets poussés sur HF Hub depuis le notebook EDA : `{username}/medical-sft-5k` et `{username}/medical-dpo-5k`

**Pipeline :** Installation → Chargement modèle → SFT → DPO → Export GGUF/HF Hub

## 1. Installation

In [6]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install --upgrade trl datasets huggingface_hub

## 2. Configuration

In [7]:
from huggingface_hub import login
login()  # Coller un token avec droit write

In [11]:
# ════════════════════════════════════════════════════════════
# CONFIGURATION — Adapter ces valeurs à votre projet
# ════════════════════════════════════════════════════════════

HF_USERNAME: str = "Maphe"  # <-- Remplacer

# Datasets HF Hub (poussés depuis le notebook EDA)
SFT_DATASET_ID: str = f"{HF_USERNAME}/medical-sft-5k"
DPO_DATASET_ID: str = f"{HF_USERNAME}/medical-dpo-5k"

# Modèle de base
MODEL_NAME: str = "unsloth/Qwen3-1.7B"
MAX_SEQ_LENGTH: int = 2048
LOAD_IN_4BIT: bool = True

# LoRA
LORA_RANK: int = 32
LORA_ALPHA: int = 32

# Entraînement SFT
SFT_EPOCHS: int = 3
SFT_BATCH_SIZE: int = 2
SFT_GRAD_ACCUM: int = 4  # effective batch = 8
SFT_LR: float = 2e-4

# Entraînement DPO
DPO_EPOCHS: int = 1
DPO_BATCH_SIZE: int = 2
DPO_GRAD_ACCUM: int = 4
DPO_LR: float = 5e-5
DPO_BETA: float = 0.1

# Export
OUTPUT_HUB_ID: str = f"{HF_USERNAME}/qwen3-1.7b-medical-finetuned"

SEED: int = 42

## 3. Chargement du modèle Qwen3-1.7B avec Unsloth

In [12]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
)

model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
trainable params: 34,865,152 || all params: 1,755,440,128 || trainable%: 1.9861


## 4. Préparation du dataset SFT

Le dataset EDA a les colonnes `instruction` et `response`.  
On les convertit au format **chat messages** attendu par Qwen3 (ChatML).

In [13]:
from datasets import load_dataset

sft_dataset = load_dataset(SFT_DATASET_ID, split="train")
print(f"SFT dataset: {len(sft_dataset)} rows")
print(f"Columns: {sft_dataset.column_names}")
print(sft_dataset[0])

README.md:   0%|          | 0.00/820 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

SFT dataset: 5000 rows
Columns: ['dataset', 'source_family', 'source_repo_id', 'source_config', 'split', 'source_id', 'language', 'task_type', 'topic', 'instruction', 'response', 'answer_key', 'answer_index']
{'dataset': 'MedQuad', 'source_family': 'MedQuad', 'source_repo_id': 'keivalya/MedQuad-MedicalQnADataset', 'source_config': None, 'split': 'train', 'source_id': 'MedQuad::train::10000', 'language': 'en', 'task_type': 'open_qa', 'topic': 'genetic changes', 'instruction': 'Answer the following medical request clearly, factually, and in a structured way.\n\nClinical context:\nnan\n\nQuestion:\nWhat are the genetic changes related to inherited thyroxine-binding globulin deficiency ?', 'response': 'Inherited thyroxine-binding globulin deficiency results from mutations in the SERPINA7 gene. This gene provides instructions for making thyroxine-binding globulin. Some mutations in the SERPINA7 gene prevent the production of a functional protein, causing TBG-CD. Other mutations reduce the a

In [14]:
# Qwen3 supporte le thinking mode. Pour le fine-tuning médical on utilise
# le non-thinking mode (réponse directe) : on encapsule <think>\n\n</think>
# avant la réponse pour désactiver le raisonnement interne.
#
# Si vous voulez conserver le thinking mode sur ~75% des exemples, commentez
# la ligne `response = ...` ci-dessous et fournissez des réponses avec
# <think>...</think> dans votre dataset.


def format_sft_to_chat(example: dict) -> dict:
    """Convertit instruction/response en messages ChatML pour Qwen3."""
    system_msg = (
        "Tu es un assistant médical expert. "
        "Réponds de manière claire, factuelle et structurée. "
        "Si la question est en anglais, réponds en anglais."
    )
    # Non-thinking mode : réponse directe sans chaîne de pensée
    response = f"<think>\n\n</think>\n\n{example['response']}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": response},
    ]
    return {"messages": messages}


sft_chat_dataset = sft_dataset.map(format_sft_to_chat, remove_columns=sft_dataset.column_names)
print(sft_chat_dataset[0]["messages"])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

[{'role': 'system', 'content': 'Tu es un assistant médical expert. Réponds de manière claire, factuelle et structurée. Si la question est en anglais, réponds en anglais.'}, {'role': 'user', 'content': 'Answer the following medical request clearly, factually, and in a structured way.\n\nClinical context:\nnan\n\nQuestion:\nWhat are the genetic changes related to inherited thyroxine-binding globulin deficiency ?'}, {'role': 'assistant', 'content': '<think>\n\n</think>\n\nInherited thyroxine-binding globulin deficiency results from mutations in the SERPINA7 gene. This gene provides instructions for making thyroxine-binding globulin. Some mutations in the SERPINA7 gene prevent the production of a functional protein, causing TBG-CD. Other mutations reduce the amount of this protein or alter its structure, resulting in TBG-PD.  Researchers have also described non-inherited forms of thyroxine-binding globulin deficiency, which are more common than the inherited form. Non-inherited thyroxine-b

In [20]:
from unsloth.chat_templates import standardize_data_formats, train_on_responses_only

# 1. Standardise le format
sft_chat_dataset = standardize_data_formats(sft_chat_dataset)

# 2. Convertit les messages en texte brut (le format attendu par le tokenizer pour le SFT)
def apply_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

sft_chat_dataset = sft_chat_dataset.map(apply_template)

# Vérification du résultat
print(sft_chat_dataset[0]["text"][:500])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

<|im_start|>system
Tu es un assistant médical expert. Réponds de manière claire, factuelle et structurée. Si la question est en anglais, réponds en anglais.<|im_end|>
<|im_start|>user
Answer the following medical request clearly, factually, and in a structured way.

Clinical context:
nan

Question:
What are the genetic changes related to inherited thyroxine-binding globulin deficiency ?<|im_end|>
<|im_start|>assistant
<think>

</think>

Inherited thyroxine-binding globulin deficiency results fro


## 5. Entraînement SFT

In [28]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling

# Configuration standard
sft_args = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRAD_ACCUM,
    learning_rate=SFT_LR,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    seed=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

# On utilise un collator standard pour eviter le check 'padding_free' d'Unsloth
# qui bloque quand packing=False
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_chat_dataset,
    dataset_text_field="text",
    data_collator=data_collator,
    args=sft_args,
)

# Masquage des instructions pour n'entrainer que sur les reponses
sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [29]:
# Vérification : les labels -100 ne couvrent que la partie prompt
import numpy as np

sample = sft_trainer.train_dataset[0]
labels = np.array(sample["labels"])
total_tokens = len(labels)
trained_tokens = int((labels != -100).sum())
print(f"Tokens totaux: {total_tokens}, entraînés (réponse): {trained_tokens}")

Tokens totaux: 222, entraînés (réponse): 135


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name}, VRAM: {gpu_stats.total_memory / 1e9:.1f} GB")
print(f"VRAM utilisée: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

sft_trainer_stats = sft_trainer.train()

print(f"\nSFT terminé en {sft_trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Loss finale: {sft_trainer_stats.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4, VRAM: 15.6 GB
VRAM utilisée: 1.56 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 3 | Total steps = 1,875
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.473708


## 6. Test du modèle après SFT

In [ ]:
from unsloth.chat_templates import get_chat_template

# Active le mode inférence
FastModel.for_inference(model)

test_messages = [
    {"role": "system", "content": "Tu es un assistant médical expert."},
    {"role": "user", "content": "Quels sont les symptômes principaux du diabète de type 2 ?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
)

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(response)

## 7. Entraînement DPO (optionnel)

Affine l'alignement du modèle via les paires préférées/rejetées d'UltraMedical-Preference.

In [ ]:
dpo_dataset = load_dataset(DPO_DATASET_ID, split="train")
print(f"DPO dataset: {len(dpo_dataset)} rows")
print(f"Columns: {dpo_dataset.column_names}")

In [ ]:
def format_dpo_to_chat(example: dict) -> dict:
    """Convertit prompt/chosen/rejected en format DPO ChatML."""
    system_msg = (
        "Tu es un assistant médical expert. "
        "Réponds de manière claire, factuelle et structurée. "
        "Si la question est en anglais, réponds en anglais."
    )
    prompt_messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": example["prompt"]},
    ]
    chosen_messages = [
        {"role": "assistant", "content": f"<think>\n\n</think>\n\n{example['chosen']}"},
    ]
    rejected_messages = [
        {"role": "assistant", "content": f"<think>\n\n</think>\n\n{example['rejected']}"},
    ]
    return {
        "prompt": prompt_messages,
        "chosen": chosen_messages,
        "rejected": rejected_messages,
    }


dpo_chat_dataset = dpo_dataset.map(
    format_dpo_to_chat,
    remove_columns=dpo_dataset.column_names,
)
print(dpo_chat_dataset[0])

In [ ]:
from trl import DPOTrainer, DPOConfig

# Repasse en mode entraînement après l'inférence de test
FastModel.for_training(model)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Unsloth gère la copie de référence automatiquement
    tokenizer=tokenizer,
    train_dataset=dpo_chat_dataset,
    args=DPOConfig(
        output_dir="./dpo_output",
        num_train_epochs=DPO_EPOCHS,
        per_device_train_batch_size=DPO_BATCH_SIZE,
        gradient_accumulation_steps=DPO_GRAD_ACCUM,
        learning_rate=DPO_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        seed=SEED,
        max_length=MAX_SEQ_LENGTH,
        max_prompt_length=MAX_SEQ_LENGTH // 2,
        beta=DPO_BETA,
    ),
)

dpo_trainer_stats = dpo_trainer.train()
print(f"\nDPO terminé en {dpo_trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Loss finale: {dpo_trainer_stats.metrics['train_loss']:.4f}")

## 8. Test final post-DPO

In [ ]:
FastModel.for_inference(model)

test_questions = [
    "Quels sont les effets secondaires courants des statines ?",
    "What is the recommended first-line treatment for hypertension?",
    "Expliquez la différence entre diabète de type 1 et type 2.",
]

for q in test_questions:
    messages = [
        {"role": "system", "content": "Tu es un assistant médical expert."},
        {"role": "user", "content": q},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=512,
        temperature=0.6, top_p=0.95, top_k=20,
    )
    reply = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {reply}")

## 9. Export et sauvegarde

Plusieurs options d'export :
- **LoRA seul** : léger, se recharge avec `PeftModel`
- **Merged 16-bit** : modèle complet fusionné
- **GGUF** : pour Ollama / llama.cpp
- **HF Hub** : push directement

In [ ]:
# Sauvegarde LoRA locale
model.save_pretrained("./qwen3-medical-lora")
tokenizer.save_pretrained("./qwen3-medical-lora")
print("LoRA sauvegardé dans ./qwen3-medical-lora")

In [ ]:
# Push du modèle fusionné 16-bit sur HF Hub
model.push_to_hub_merged(
    OUTPUT_HUB_ID,
    tokenizer=tokenizer,
    save_method="merged_16bit",
    token=True,
    private=True,
)
print(f"Modèle fusionné poussé → https://huggingface.co/{OUTPUT_HUB_ID}")

In [ ]:
# (Optionnel) Export GGUF pour Ollama / llama.cpp
# Décommenter pour exporter en Q4_K_M (bon compromis taille/qualité)

# model.push_to_hub_gguf(
#     f"{OUTPUT_HUB_ID}-GGUF",
#     tokenizer=tokenizer,
#     quantization_method="q4_k_m",
#     token=True,
#     private=True,
# )
# print(f"GGUF poussé → https://huggingface.co/{OUTPUT_HUB_ID}-GGUF")